In [1]:
!pip -q install sentence-transformers
!pip -q install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.1 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import faiss
import pickle

from sentence_transformers import SentenceTransformer

In [5]:
df = pd.read_csv("/content/processed_chunks.csv")

print(df.shape)

df.head()

(9723, 3)


,source,chunk_id,text
0,amt_powerplant_handbook.pdf,0,Aviation Maintenance Technician \nHandbook–Pow...
1,amt_powerplant_handbook.pdf,1,in the Aviation Maintenance Technician Handboo...
2,amt_powerplant_handbook.pdf,2,manufacturer's manuals or other textbooks if m...
3,amt_powerplant_handbook.pdf,3,"Testing Standards Branch, AFS-630, P.O. Box 25..."
4,amt_powerplant_handbook.pdf,4,Teledyne Continental Motors (www.genuinecontin...


In [6]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
texts = df["text"].fillna("").astype(str).tolist()

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding Shape:", embeddings.shape)

Batches:   0%|          | 0/304 [00:00<?, ?it/s]

Embedding Shape: (9723, 384)


In [9]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 9723


In [10]:
faiss.write_index(index, "aeroassist_index.faiss")

print("FAISS index saved successfully!")

FAISS index saved successfully!


In [11]:
import pickle

metadata = df.to_dict(orient="records")

with open("metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Metadata saved successfully!")

Metadata saved successfully!


In [12]:
from google.colab import files

files.download("aeroassist_index.faiss")
files.download("metadata.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
query = "How is corrosion inspected in aircraft?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

distances, indices = index.search(query_embedding, k=5)

for rank, idx in enumerate(indices[0], start=1):
    print("=" * 100)
    print(f"Result {rank}")
    print(f"Source: {metadata[idx]['source']}")
    print()
    print(metadata[idx]["text"][:1000])
    print()

Result 1
Source: FAA-H-8083-31B_Aviation_Maintenance_Technician_Handbook.pdf

• 
Cracks should not intersect, which creates the 
potential for the loss of a portion of a head. 
Inspection for Corrosion 
Corrosion is the gradual deterioration of metal due to a 
chemical or electrochemical reaction with its environment. 
The reaction can be triggered by the atmosphere, moisture, 
or other agents. When inspecting the structure of an aircraft, 
it is important to watch for evidence of corrosion on both the 
outside and inside. Corrosion on the inside is most likely to 
occur in pockets and corners where moisture and salt spray 
may accumulate; therefore, drain holes must always be kept 
clean. Also inspect the surrounding members for evidence 
of corrosion. 
Damage Removal 
To prepare a damaged area for repair: 
1. 
Remove all distorted skin and structure in damaged 
area. 
Figure 4-172. Smoking rivet. 
2. 
Remove damaged material so that the edges of the 
completed repair match existing s